# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---


## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `
orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets

print(orders.shape)
print(orders.dtypes)
print(orders.head())
orders.info()
print()
print(orders.isnull().sum())
print(orders.duplicated().sum())

(25100, 12)
id_pedido              object
id_usuario             object
fecha_hora_pedido      object
pais                   object
dispositivo            object
fuente_referencia      object
nombre_producto        object
categoria_producto     object
cantidad              float64
precio_unitario       float64
monto_descuento       float64
monto_total           float64
dtype: object
  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica   

**Orders**
- Tipo de dato erróneo en `fecha_hora_pedido` (datetime) y `cantidad` ()
- 100 filas duplicadas
- nulos en `pais` (300), `dispositivo` (20), `categoria_producto` (80), `fuente_referencia` (30), `nombre_producto` (30), `cantidad` (50), `precio_unitario` (50), `monto_descuento` (50)

In [4]:
print(catalog.shape)
print(catalog.dtypes)
print(catalog.head())
catalog.info()
print()
print(catalog.isnull().sum())
print(catalog.duplicated().sum())

(7, 4)
nombre_producto        object
categoria_producto     object
costo_unitario        float64
proveedor              object
dtype: object
        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finley  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unita

**Catalog**
- Corregir Electrónica a Electronica (como está en `orders`)

In [5]:
print(marketing.shape)
print(marketing.dtypes)
print(marketing.head())
marketing.info()
print()
print(marketing.isnull().sum())
print(marketing.duplicated().sum())

(1620, 5)
fecha          object
pais           object
id_campaña     object
canal          object
gasto         float64
dtype: object
        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
2  2025-01-01    Mexico         social_Mexico       social  2045.01
3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB

fecha      

**Marketing**
- Tipo de dato erróneo en `fecha`
- `canal`: 101 nulos

---


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 


---

In [6]:
# --- ORDERS ---

# Eliminamos duplicados
orders = orders.drop_duplicates()

# Formato fechas
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])

# Nulos en columnas críticas
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_descuento'])
orders = orders.dropna(subset=['nombre_producto'])

# Convertir cantidad a int
orders['cantidad'] = orders['cantidad'].astype(int)

# Nulos en categóricas
orders[['pais', 'dispositivo', 'fuente_referencia']] = orders[['pais', 'dispositivo', 'fuente_referencia']].fillna('Unknown')

# Valores negativos
orders = orders[orders['cantidad'] > 0]

# Consistencia de montos
orders['monto_total'] = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']

# Estandarizar países
orders['pais'] = orders['pais'].str.title()

# Verificación final
print(orders.shape)
print(orders.isnull().sum())
print(orders['pais'].value_counts())

(24916, 12)
id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64
Mexico       8313
Colombia     8282
Argentina    8025
Unknown       296
Name: pais, dtype: int64


In [7]:
# --- CATALOG ---

# Estandarizar Electronica
catalog['categoria_producto'] = catalog['categoria_producto'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

# Verificación
print(catalog['categoria_producto'].value_counts())


Electronica    3
Hogar          2
Moda           2
Name: categoria_producto, dtype: int64


In [8]:
# --- LIMPIEZA: MARKETING ---

# Formato fechas
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# Nulos en canal
marketing['canal'] = marketing['canal'].fillna('Unknown')

# Verificación
print(marketing.dtypes)
print(marketing.isnull().sum())
print(marketing['canal'].value_counts())

fecha         datetime64[ns]
pais                  object
id_campaña            object
canal                 object
gasto                float64
dtype: object
fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64
paid_search    507
social         506
organic        506
Unknown        101
Name: canal, dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [9]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---


## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [10]:
# Unión de datasets

orders_catalog = orders.merge(catalog, on=['nombre_producto', 'categoria_producto'], how='inner')
print(orders_catalog.shape)
print(orders_catalog.columns.tolist())
print(orders_catalog.head())

(24916, 14)
['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'costo_unitario', 'proveedor']
  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1  order_12  user_6810        2025-03-17   Colombia      mobile   
2  order_13  user_3170        2025-04-10   Colombia     desktop   
3  order_15  user_1500        2025-04-03   Colombia      mobile   
4  order_19  user_7190        2025-02-03  Argentina      mobile   

  fuente_referencia  nombre_producto categoria_producto  cantidad  \
0           organic  Jacket-Winter-M               Moda         2   
1            social  Jacket-Winter-M               Moda         1   
2           organic  Jacket-Winter-M               Moda         1   
3           organic  Jacket-Winter-M               Moda         2   
4         

In [ ]:
revenue_total = orders_catalog['monto_total'].sum()
print("Revenue total:", round(revenue_total, 2))
costo_total = (orders_catalog['cantidad'] * orders_catalog['costo_unitario']).sum()
print("Costo total:", round(costo_total, 2))
inversion_marketing = marketing['gasto'].sum()
print("Inversion en Marketing:", round(inversion_marketing, 2))
profit = (revenue_total - costo_total - inversion_marketing)
print("Profit:", round(profit, 2))

Revenue total: 51954719.78
Costo total: 43124069.01
Inversion en Marketing: 2871843.53
Profit: 5958807.24


In [ ]:
ticket_promedio = orders_catalog['monto_total'].mean()
print("Ticket promedio:", round(ticket_promedio, 2))
cantidad_promedio = orders_catalog['cantidad'].mean()
print("Cantidad promedio de productos:", round(cantidad_promedio, 2))
producto_mas_vendido = orders_catalog.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False).head(1)
print("Producto más vendido:", producto_mas_vendido)
gasto_marketing = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print("Gasto Marketing por canal:", gasto_marketing)

Ticket promedio: 2085.2
Cantidad promedio de productos: 7.12
Producto más vendido: nombre_producto
Laptop-Gaming-16GB    144198
Name: cantidad, dtype: int64
Gasto Marketing por canal: canal
social         918043.21
organic        913533.01
paid_search    863088.21
Unknown        177179.10
Name: gasto, dtype: float64


---


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'YOUR_PASSWORD_HERE', #credenciales de entorno TripleTen
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [14]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [15]:
query_eventos = '''
SELECT DISTINCT nombre_evento
FROM events;
'''
pd.read_sql(query_eventos, con=engine)

,nombre_evento
0,add_payment_info
1,first_visit
2,begin_checkout
3,add_to_cart
4,select_item
5,purchase


In [16]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT nombre_evento, COUNT(DISTINCT id_usuario)
FROM events
GROUP BY nombre_evento
ORDER BY CASE nombre_evento
    WHEN 'first_visit' THEN 1
    WHEN 'select_item' THEN 2
    WHEN 'add_to_cart' THEN 3
    WHEN 'begin_checkout' THEN 4
    WHEN 'add_payment_info' THEN 5
    WHEN 'purchase' THEN 6
END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,count
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [17]:

# PARTE 2: Conversiones
# ======================


query_conversion = '''
WITH cte_first_visit AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'first_visit'
),
cte_select_item AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'select_item'
),
cte_add_to_cart AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_to_cart'
),
cte_begin_checkout AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'begin_checkout'
),

cte_add_payment_info AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'add_payment_info'
),
cte_purchase AS (
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento = 'purchase'
)
SELECT 
    (SELECT COUNT (*) FROM cte_first_visit) AS first_visit_users,
    (SELECT COUNT (*) FROM cte_select_item) AS select_item_users,
    (SELECT COUNT (*) FROM cte_add_to_cart) AS add_to_cart_users,
    (SELECT COUNT (*) FROM cte_begin_checkout) AS begin_checkout_users,
    (SELECT COUNT (*) FROM cte_add_payment_info) AS add_payment_info_users,
    (SELECT COUNT (*) FROM cte_purchase) AS purchase_users,
    

ROUND(((SELECT COUNT(*) FROM cte_first_visit)-(SELECT COUNT(*) FROM cte_select_item))*100.0/NULLIF((SELECT COUNT (*) FROM cte_first_visit), 0), 2) AS dropoff_after_first_visit_pct,
ROUND(((SELECT COUNT(*) FROM cte_select_item)-(SELECT COUNT(*) FROM cte_add_to_cart))*100.0/NULLIF((SELECT COUNT (*) FROM cte_select_item), 0), 2) AS dropoff_after_select_item_pct,
ROUND(((SELECT COUNT(*) FROM cte_add_to_cart)-(SELECT COUNT(*) FROM cte_begin_checkout))*100.0/NULLIF((SELECT COUNT (*) FROM cte_add_to_cart), 0), 2) AS dropoff_after_add_to_cart_pct,
ROUND(((SELECT COUNT(*) FROM cte_begin_checkout)-(SELECT COUNT(*) FROM cte_add_payment_info))*100.0/NULLIF((SELECT COUNT (*) FROM cte_begin_checkout), 0), 2) AS dropoff_after_begin_checkout_pct,
ROUND(((SELECT COUNT(*) FROM cte_add_payment_info)-(SELECT COUNT(*) FROM cte_purchase))*100.0/NULLIF((SELECT COUNT (*) FROM cte_add_payment_info), 0), 2) AS dropoff_after_add_payment_info_pct,

ROUND((SELECT COUNT(*) FROM cte_purchase) * 100.0 / NULLIF((SELECT COUNT(*) FROM cte_first_visit), 0), 2) AS conversion_rate_final_pct;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,first_visit_users,select_item_users,add_to_cart_users,begin_checkout_users,add_payment_info_users,purchase_users,dropoff_after_first_visit_pct,dropoff_after_select_item_pct,dropoff_after_add_to_cart_pct,dropoff_after_begin_checkout_pct,dropoff_after_add_payment_info_pct,conversion_rate_final_pct
0,7796,7582,7634,7208,6250,6240,2.74,-0.69,5.58,13.29,0.16,80.04


De los 7.796 usuarios que visitaron la plataforma, solo el 80% completó una compra. El mayor punto de abandono se encuentra en la etapa de begin_checkout, donde se pierde el 13,29% de los usuarios.

---


## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [18]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [19]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [20]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes AS (
    SELECT 
        u.id_usuario,
        TO_CHAR(CAST(u.fecha_registro AS DATE), 'YYYY-MM') AS cohorte,
        ua.dias_despues_registro,
        ua.activo
    FROM users u
    JOIN user_activity ua ON u.id_usuario = ua.id_usuario
),
retencion AS (
    SELECT cohorte,
    COUNT (*) AS clientes_iniciales,
    COUNT(CASE WHEN dias_despues_registro BETWEEN 1 AND 7 AND activo = 1 THEN 1 END) AS retenido_w1,
    COUNT(CASE WHEN dias_despues_registro BETWEEN 8 AND 14 AND activo = 1 THEN 1 END) AS retenido_w2,
    COUNT(CASE WHEN dias_despues_registro BETWEEN 15 AND 21 AND activo = 1 THEN 1 END) AS retenido_w3
    FROM cohortes
    GROUP BY cohorte
)
SELECT cohorte, clientes_iniciales,
ROUND(retenido_w1::numeric/clientes_iniciales *100, 2) AS semana_1,
ROUND(retenido_w2::numeric/clientes_iniciales *100, 2) AS semana_2,
ROUND(retenido_w3::numeric/clientes_iniciales *100, 2) AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,clientes_iniciales,semana_1,semana_2,semana_3
0,2025-01,6508,10.71,10.26,10.08
1,2025-02,5776,10.58,10.54,10.99
2,2025-03,6544,10.35,10.77,10.54
3,2025-04,6424,10.59,10.85,10.32
4,2025-05,6748,10.30,10.02,10.46


Analizando la retención por cohortes, se observa que aproximadamente el 10% de los usuarios permanece activo en las semanas 1, 2 y 3 tras su registro, sin diferencias significativas entre cohortes.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay diferencia en la tasa de conversión entre el grupo control y el grupo tratamiento
   - **H₁ (Hipótesis alternativa):** El grupo tratamiento tiene una tasa de conversión diferente al grupo control
   
**Test estadístico:** Z-test de proporciones
**Nivel de significancia alpha:** 0.05

In [21]:
experiment=pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
print(experiment.shape)
print(experiment.head())
print(experiment.groupby('variante')['convirtio'].mean())

(10000, 7)
   id_usuario     variante  convirtio dispositivo       pais  duracion_sesion  \
0  exp_user_0  tratamiento          0      mobile  Argentina           114.41   
1  exp_user_1  tratamiento          0     desktop     Mexico           170.03   
2  exp_user_2      control          1      mobile   Colombia           140.21   
3  exp_user_3  tratamiento          0      mobile   Colombia           151.45   
4  exp_user_4  tratamiento          0     desktop     Mexico           299.96   

    timestamp  
0  2025-03-28  
1  2025-01-15  
2  2025-03-18  
3  2025-06-03  
4  2025-01-12  
variante
control        0.156898
tratamiento    0.162860
Name: convirtio, dtype: float64


In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

conversiones = experiment.groupby('variante')['convirtio'].sum()
totales = experiment.groupby('variante')['convirtio'].count()

exitos = [conversiones['tratamiento'], conversiones['control']]
observaciones = [totales['tratamiento'], totales['control']]

z_stat, p_value = proportions_ztest(exitos, observaciones)
print(f"Estadístico z: {z_stat}")
print(f"Valor p: {p_value}")

alpha = 0.05
if p_value < alpha:
    print("Rechazamos la hipótesis nula: hay evidencia de una diferencia.")
else:
    print("No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.")


Estadístico z: 0.8132782986429474
Valor p: 0.41605851639119995
No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia.


La diferencia observada entre el grupo control (15.7%) y tratamiento (16.3%) no es estadísticamente significativa con un p-value de 0.42. Esto significa que el cambio en la UI del checkout no tuvo un impacto demostrable en la tasa de conversión.

**Recomendación para el negocio:** no implementar el cambio de UI basándose en estos resultados.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

*Se construyeron 2 dashboards interactivos en Tableau: un Overview Ejecutivo con KPIs principales y un Detalle con drill-through por producto. Links a continuación.*
- Overview: https://public.tableau.com/views/RappiPlus-Overview/OVERVIEW?:language=en-GB&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
- Detalle: https://public.tableau.com/views/RappiPlus-Detalle/DETALLE?:language=en-GB&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

---